# M3: fine-tune whisper-small on AfriSpeech-200 (Kaggle)

Run with **Save Version → Save & Run All (Commit)** so it keeps going after you close the browser.
Settings: Accelerator **GPU T4 x2**, Internet **on**. Steps: `knowledge/how-to-train-on-kaggle.md`.

Everything this run produces lands in `/kaggle/working/` (the notebook's Output tab):
- `m3-whisper-small-afrispeech/`: proof to commit (`train.log`, `train_log.csv`, `stdout-*.log`, environment files, and after prediction `predictions.csv`, `predict.log`, `scores.json`)
- `checkpoints/whisper-small-afrispeech/`: weights (`best/`, and `last/` to resume). **Never commit these.**

To resume after a session stops: add this notebook's previous version output as Input, then run again.

In [ ]:
# 1. Check the machine: GPU, disk. Saved to the log folder as proof.
!mkdir -p /kaggle/working/m3-whisper-small-afrispeech
!(date -u; nvidia-smi; df -h /kaggle/working /tmp) | tee /kaggle/working/m3-whisper-small-afrispeech/machine-$(date -u +%Y%m%dT%H%M%S).log

In [ ]:
# 2. Get the code at the branch, install the locked dependencies (Python 3.12, torch with CUDA)
BRANCH = "6-whisper"
!rm -rf /tmp/repo && git clone --quiet --branch {BRANCH} https://github.com/ASU-MICS-2028/nlp-group-1-prosit-3.git /tmp/repo
!pip install --quiet uv
!cd /tmp/repo && git log -1 --format='commit %H %s' && uv sync --frozen --quiet && uv run python -c "import torch; print('cuda', torch.cuda.is_available(), torch.__version__)"

In [ ]:
# 3. Point the repo's output folders at /kaggle/working, so they are saved even if a later cell fails.
#    If a previous session's output is attached as Input, copy it in to resume.
import glob, os, shutil
RUN = "whisper-small-afrispeech"
W = "/kaggle/working"
os.makedirs(f"{W}/checkpoints", exist_ok=True)
for prev in glob.glob(f"/kaggle/input/*/m3-{RUN}") + glob.glob(f"/kaggle/input/*/*/m3-{RUN}"):
    base = os.path.dirname(prev)
    print("resuming from", base)
    shutil.copytree(prev, f"{W}/m3-{RUN}", dirs_exist_ok=True)
    if os.path.isdir(f"{base}/checkpoints/{RUN}"):
        shutil.copytree(f"{base}/checkpoints/{RUN}", f"{W}/checkpoints/{RUN}", dirs_exist_ok=True)
    break
!cd /tmp/repo && ln -sfn {W}/checkpoints checkpoints && ln -sfn {W}/m3-{RUN} eval/results/m3-{RUN} && ls -la eval/results | tail -3

In [ ]:
# 4. Train. Stops cleanly after --max-hours and saves state; run the notebook again (step 3) to resume.
#    train.log and train_log.csv are written by the script; stdout-*.log also keeps warnings and errors.
!cd /tmp/repo && uv run python -m models.whisper_full.finetune --model openai/whisper-small --name {RUN} \
    --steps 4000 --batch 8 --accumulate 2 --train-files 14 --max-hours 10 \
    2>&1 | tee -a {W}/m3-{RUN}/stdout-train-$(date -u +%Y%m%dT%H%M%S).log

In [ ]:
# 5. Only when training finished: predict the 1,000 test clips with the best checkpoint, then score.
done = "=== done" in open(f"{W}/m3-{RUN}/train.log").read()
print("training finished:", done)
if done:
    !cd /tmp/repo && uv run python -m models.whisper_full.zero_shot checkpoints/{RUN}/best --fallback 2>&1 | tee -a {W}/m3-{RUN}/stdout-predict.log
    !cd /tmp/repo && uv run python -m common.score eval/results/m3-{RUN}/predictions.csv 2>&1 | tee -a {W}/m3-{RUN}/stdout-score.log

In [ ]:
# 6. What was saved (checkpoint sizes, and the proof files to download and commit)
!du -sh {W}/checkpoints/{RUN}/* 2>/dev/null; ls -la {W}/m3-{RUN}; tail -5 {W}/m3-{RUN}/train.log